# 06 — HMM + Viterbi route reconstruction (Member C: nisarg-ravi)
Milestone M7. Needs the outputs of `05_bayes_opt.ipynb`.

* **Hidden state** at GPS point t = one candidate arc of that point. The candidate row keeps its arc id, so the decoded state *is* a road segment.
* **Emission** = log of the tuned LightGBM probability, normalised over that point's candidates.
* **Transition** = −|driving distance − GPS straight-line distance| / β (Newson & Krumm 2009). Driving distance uses a bounded Dijkstra on the directed graph.
* **Viterbi**: δ_t(j) = e_t(j) + max_i [δ_{t−1}(i) + a(i, j)], keep back-pointers, then backtrack.
* **Edge cases**: gap > `gap_s` → chain split; no path → fixed `unreachable` penalty (chain never dies); empty candidate sets cannot occur (D7).
* Decoder parameters are tuned on **train with out-of-fold probabilities** only, then applied once to student-test.

In [ ]:
# ---- Config (edit if your paths differ) ----
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/paya5am/01_pathsnap.git"
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ZIP = "/content/drive/MyDrive/PathSnap/Student folder.zip"
    if not os.path.exists("/content/01_pathsnap"):
        subprocess.run(["git", "clone", REPO_URL, "/content/01_pathsnap"], check=True)
    os.chdir("/content/01_pathsnap/alt_lightgbm_pipeline")   # self-contained sub-project (own pathsnap)
    if not os.path.exists("data/train"):
        os.makedirs("data", exist_ok=True)
        subprocess.run(["unzip", "-q", "-o", ZIP, "-d", "/content/raw"], check=True)
        subprocess.run("cp -r '/content/raw/Student folder/train' '/content/raw/Student folder/student_test' data/", shell=True, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm", "optuna", "folium"], check=True)
else:
    os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
sys.path.insert(0, os.getcwd())
DATA = "data"                       # contains train/ and student_test/
DATASET = "experiments/dataset"     # candidate tables
RESULTS = "experiments/results"     # all logged outputs of this notebook
FIGS = "reports/figures/c_models"
for d in [DATASET, RESULTS, FIGS]:
    os.makedirs(d, exist_ok=True)
print("working dir:", os.getcwd())

In [ ]:
import subprocess, sys, json, pandas as pd
if not os.path.exists(f'{RESULTS}/hmm_vs_classifier_summary.csv'):
    subprocess.run([sys.executable, 'scripts/run_hmm.py', DATA, RESULTS], check=True)
print(json.load(open(f'{RESULTS}/hmm_config.json')))
S = pd.read_csv(f'{RESULTS}/hmm_vs_classifier_summary.csv'); S.round(4)

## Decoder parameter search (train, out-of-fold)

In [ ]:
import matplotlib.pyplot as plt
g = pd.read_csv(f'{RESULTS}/hmm_grid_train.csv')
piv = g.pivot(index='unreachable', columns='beta', values='train_oof_top1')
fig, ax = plt.subplots(figsize=(7, 3))
im = ax.imshow(piv.values, aspect='auto', cmap='viridis')
ax.set_xticks(range(len(piv.columns)), piv.columns); ax.set_yticks(range(len(piv.index)), piv.index)
ax.set_xlabel('beta (m)'); ax.set_ylabel('unreachable penalty')
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        ax.text(j, i, f'{piv.values[i,j]:.4f}', ha='center', va='center', color='w', fontsize=7)
plt.colorbar(im, label='train OOF Top-1'); plt.title('HMM grid search'); plt.tight_layout()
plt.savefig(f'{FIGS}/hmm_grid.png', dpi=150); plt.show()
pd.read_csv(f'{RESULTS}/hmm_gap_sensitivity_train.csv')

## Classifier-only vs classifier + HMM, per student-test trip

In [ ]:
P = pd.read_csv(f'{RESULTS}/hmm_vs_classifier_per_trip.csv')
st = P[P.partition == 'student_test'].pivot(index='trip_id', columns='method', values=['top1', 'continuity', 'stitched_seq_f1'])
ax = st['top1'].plot.bar(figsize=(9, 3.5)); ax.set_ylabel('Top-1 (labelled points)'); ax.set_ylim(0.6, 1)
plt.title('Student-test: classifier vs classifier+HMM'); plt.tight_layout(); plt.savefig(f'{FIGS}/hmm_vs_clf_student_test.png', dpi=150); plt.show()
st.round(4)

## Folium map: one student-test trip (reference vs classifier vs HMM)
The reference is drawn only for the report. The predictor never reads `.route`.

In [ ]:
import folium, numpy as np
from pathsnap.inference import load_artifacts, predict_trip_folder
from pathsnap.io import load_trip_folder
from pathsnap import hmm as H
import shutil
ART = 'artifacts'
if not os.path.exists(f'{ART}/model.joblib'):
    subprocess.run([sys.executable, 'scripts/export_artifacts.py', RESULTS, ART], check=True)
arts = load_artifacts(ART)
TRIP = 76   # student-test trip with a reference-route break
folder = f'{DATA}/student_test/{TRIP:08d}'
pred = predict_trip_folder(folder, arts)
trip = load_trip_folder(folder)
prep = pred['_prep']; net = prep['net']
def lines(arcs):
    return [[(trip.nodes[net.frm[a],1], trip.nodes[net.frm[a],0]), (trip.nodes[net.to[a],1], trip.nodes[net.to[a],0])] for a in arcs]
m = folium.Map(location=[trip.track[:,1].mean(), trip.track[:,0].mean()], zoom_start=15, tiles='cartodbpositron')
fg = {k: folium.FeatureGroup(name=k) for k in ['Reference route', 'Classifier only', 'Classifier + HMM (stitched)', 'GPS points']}
for seg in lines(trip.route): folium.PolyLine(seg, color='black', weight=7, opacity=0.4).add_to(fg['Reference route'])
clf_route = H.stitch_route(np.array(pred['classifier_point_arcs']), prep['t'], net, prep['px'], prep['py'])
for seg in lines(clf_route): folium.PolyLine(seg, color='orange', weight=3).add_to(fg['Classifier only'])
for seg in pred['route_latlon_segments']: folium.PolyLine(seg, color='blue', weight=3).add_to(fg['Classifier + HMM (stitched)'])
for lat, lon in pred['gps_latlon'][::5]: folium.CircleMarker((lat, lon), radius=1.5, color='red').add_to(fg['GPS points'])
for f in fg.values(): f.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
os.makedirs('reports/maps/c_hmm', exist_ok=True)
m.save(f'reports/maps/c_hmm/trip_{TRIP}_hmm_vs_classifier.html'); m

## M8 — Export artifacts for the API
`artifacts/` = `model.joblib` (tuned LightGBM), `preprocessing_config.json` (candidate rule, feature list), `hmm_config.json` (tuned decoder), `manifest.json` (params, SHA-256 hashes, library versions).

D's FastAPI only needs `pathsnap.inference.load_artifacts()`, `predict_trip_folder()` (never reads `.route`) and `evaluate_prediction()` (server-side scoring, run after prediction).

In [ ]:
import json, shutil, tempfile, time
subprocess.run([sys.executable, 'scripts/export_artifacts.py', RESULTS, ART], check=True)
print(json.dumps(json.load(open(f'{ART}/manifest.json')), indent=1))
arts = load_artifacts(ART)
# Leakage check: copy a trip WITHOUT its .route file and predict on it
tmp = tempfile.mkdtemp(); src = f'{DATA}/student_test/00000000'
shutil.copytree(src, f'{tmp}/00000000'); os.remove(f'{tmp}/00000000/00000000.route')
t0 = time.time(); pred = predict_trip_folder(f'{tmp}/00000000', arts)
print('predicted without .route:', pred['n_points'], 'points,', len(pred['route_arcs']), 'route arcs in %.2f s' % (time.time()-t0))
from pathsnap.inference import evaluate_prediction
print('server-side score:', evaluate_prediction(pred, src))